# Milestone 1 — Healthcare Liver Disease Decision Support
**Solution level:** Core 50

Core solution (~50-level): completes the required workflow with clear baseline analysis. It is intentionally concise; add deeper interpretation and validation for a higher score.

**Dataset files expected:** `data/Liver Patient Dataset.csv` and `data/Status.xlsx`.

**Target definition:** Status 1 = liver patient; Status 2 = non-liver patient. This is an educational exercise, not a diagnostic tool.

## 1. Load and inspect clinical data

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

DATA_DIR = Path("data")
patients = pd.read_csv(DATA_DIR / "Liver Patient Dataset.csv")
status = pd.read_excel(DATA_DIR / "Status.xlsx")
print("Patient data:", patients.shape, "| Status file:", status.shape)
print("Patient columns:", patients.columns.tolist())
print("Status columns:", status.columns.tolist())
display(patients.head())
display(status.head())

## 2. Verify target and join if required

In [ ]:
# Common Liver Patient Dataset releases already contain the Status target.
# If not, join Status.xlsx after verifying the matching patient identifier.
df = patients.copy()
if "Status" not in df.columns:
    possible_keys = [c for c in ["ID", "Patient_ID", "PatientID"] if c in df.columns and c in status.columns]
    if not possible_keys:
        raise ValueError("Status is not in patient data. Inspect both files and set the verified common key.")
    key = possible_keys[0]
    df = df.merge(status, on=key, how="left", validate="one_to_one")
if "Status" not in df.columns:
    raise ValueError("Could not locate Status after loading/joining.")
df["Status"] = pd.to_numeric(df["Status"], errors="coerce")
df = df[df["Status"].isin([1,2])].copy()
print("Joined/usable data:", df.shape)
print("Target counts (1=Liver patient, 2=Non-liver patient):")
display(df["Status"].value_counts(dropna=False))
display(df.isna().sum().sort_values(ascending=False).head(15))
print("Duplicate rows:", df.duplicated().sum())

## 3. Exploratory analysis

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

sns.countplot(data=df, x="Status")
plt.title("Target class distribution (1 = liver patient, 2 = non-liver patient)")
plt.show()
clinical_cols = [c for c in ["Age","TB","DB","Alkphos","Sgpt","Sgot","TP","ALB","A/G Ratio"] if c in df.columns]
for col in clinical_cols:
    fig, ax = plt.subplots(figsize=(6,3))
    sns.histplot(df[col], kde=True, ax=ax)
    ax.set_title(f"Distribution of {col}")
    plt.show()

## 4. Logistic Regression baseline

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, accuracy_score

y = df["Status"].map({1:1, 2:0}).astype(int)
X = df.drop(columns=["Status", "ID", "Patient_ID", "PatientID"], errors="ignore")
cat_cols = X.select_dtypes(include=["object","category"]).columns.tolist()
num_cols = X.select_dtypes(include=np.number).columns.tolist()
prep = ColumnTransformer([
    ("num", Pipeline([("imputer", SimpleImputer(strategy="median")), ("scale", StandardScaler())]), num_cols),
    ("cat", Pipeline([("imputer", SimpleImputer(strategy="most_frequent")),
                      ("onehot", OneHotEncoder(handle_unknown="ignore"))]), cat_cols)
])
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
model = Pipeline([("prep", prep), ("classifier", LogisticRegression(max_iter=2000, class_weight="balanced"))])
model.fit(X_train, y_train)
pred = model.predict(X_test)
prob = model.predict_proba(X_test)[:,1]
print("Accuracy:", accuracy_score(y_test, pred))
print(classification_report(y_test, pred, zero_division=0))
print("Confusion matrix:\n", confusion_matrix(y_test, pred))
try:
    print("ROC-AUC:", roc_auc_score(y_test, prob))
except ValueError as e:
    print("ROC-AUC unavailable:", e)

## 5. Responsible interpretation
- Prioritize recall and false-negative review for the positive class.
- Report precision, recall, F1, confusion matrix and ROC-AUC where calculable.
- Do not interpret model output as diagnosis or prescribe clinical action.
- Missingness, sample size, class imbalance and population representativeness limit reliability.